# Non-Contractual Player Churn Analytics & Predictive Pipeline

## 1. Business Understanding & Problem Formulation

What do you think the cost of acquiring a new customer for betting firms is? 
A whooping [$3.9 Billion](https://www.5wpr.com/new/the-gambling-industry-spent-3-9-billion-on-marketing-last-year-here-is-what-the-data-shows/) spread affross different advertisement portfolios to get users to walk through their front door. However, they are all faced with one problem, after filling in the last registration form, getting users to continuously place bets has become another matter entirely. [Data](https://igamingbusiness.com/marketing-affiliates/dot-player-churn-a-real-problem/) suggests only 63% of users active for more than 2 years, and only a third of new users make it to the third year still using betting services.   

The aim of this process is to better understand user behavior through rigorous exploratory analysis and model customer behaviour to predict user churning and possibly reduce churning. 

For subscription based services like Netflix, a churn can be defined as a user cancelling their subscription, usually days in advance with a physical action of clicking an unsubscribe button. 
However, this is not the case for betting companies. A user's churn is usually marked by abandoning an account and ceasing to place bets, which does very little in announcing depature. 

In this study, churn is defined as a player's failure to place a bet or make a deposit for a 45-day target period following a fixed observation cutoff date ($T_{\text{cutoff}}$). This 45-day window represents a valid business compromise, accounting for the fact that active players often pause betting due to normal day-to-day life events or seasonal sports breaks. Using $T_{\text{cutoff}}$ as our target boundary, player behavior leading up to departure is analyzed and the factors driving disengagement are modeled.

The formula is as follows

$$Y_i = \begin{cases} 1 & \text{if } \sum_{t = T_{\text{cutoff}}}^{T_{\text{cutoff}} + 45} \left(\text{Bets}_{i,t} + \text{Deposits}_{i,t}\right) = 0 \\ 0 & \text{otherwise} \end{cases}$$

## Data Understanding

In this phase, the size, memory requirements, cardinality, temporal and logical anomalies are identified. 


In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import duckdb

In [2]:
# Connecting to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

In [3]:
# Grouping the datasets into a dictionary for easy access
data = {
    'Demographics': 'datasets/SecondSeshData1Demographics.csv',
    'CashGames': 'datasets/SecondSeshData2CashGames.csv',
    'Tournaments': 'datasets/SecondSeshData3Tournaments.csv',
    'Deposits': 'datasets/SecondSeshData4Deposits.csv',
    'Withdrawals': 'datasets/SecondSeshData5Withdrawals.csv',
}

In [4]:
for table_name, file_path in data.items():
    query = f"CREATE OR REPLACE VIEW {table_name} AS SELECT * FROM read_csv_auto('{file_path}')"
    con.execute(query)

print("All CSV views registered successfully in DuckDB!")

All CSV views registered successfully in DuckDB!


In [5]:
# See all registered views
con.execute("SHOW TABLES").fetchdf()

,name
0,CashGames
1,Demographics
2,Deposits
3,Tournaments
4,Withdrawals


In [6]:
# Check shape,number of users, and min/max dates for each table
audit_query = """
SELECT 
    'Demographics' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT UserID) AS unique_users,
    NULL AS min_date,
    NULL AS max_date
FROM Demographics

UNION ALL

SELECT 
    'CashGames',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM CashGames

UNION ALL

SELECT 
    'Tournaments',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(Date AS TIMESTAMP)),
    MAX(CAST(Date AS TIMESTAMP))
FROM Tournaments

UNION ALL

SELECT 
    'Deposits',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Deposits

UNION ALL

SELECT 
    'Withdrawals',
    COUNT(*),
    COUNT(DISTINCT UserID),
    MIN(CAST(ProcessDate AS TIMESTAMP)),
    MAX(CAST(ProcessDate AS TIMESTAMP))
FROM Withdrawals;
"""

# Run the query and fetch into a Pandas dataframe
audit_results = con.execute(audit_query).fetchdf()
audit_results

,table_name,total_rows,unique_users,min_date,max_date
0,Demographics,5028,5028,NaT,NaT
1,CashGames,51763,4232,2015-02-01,2020-07-28
2,Tournaments,82831,3448,2015-02-01,2021-01-01
3,Deposits,295088,5008,2015-02-01,2020-07-15
4,Withdrawals,32307,1827,2015-02-01,2020-07-15


In [7]:
# Are there transacting users missing from Demographics? If so, how many?
orphaned_users_query = """
-- Check for transaction users missing from Demographics
SELECT 'Orphan CashGame Users' AS issue_type, COUNT(DISTINCT UserID) AS missing_count
FROM CashGames 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Deposit Users', COUNT(DISTINCT UserID)
FROM Deposits 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Withdrawal Users', COUNT(DISTINCT UserID)
FROM Withdrawals 
WHERE UserID NOT IN (SELECT UserID FROM Demographics)

UNION ALL

SELECT 'Orphan Tournament Users', COUNT(DISTINCT UserID)
FROM Tournaments 
WHERE UserID NOT IN (SELECT UserID FROM Demographics);
"""
orphaned_users_results = con.execute(orphaned_users_query).fetchdf()
orphaned_users_results

,issue_type,missing_count
0,Orphan CashGame Users,0
1,Orphan Deposit Users,0
2,Orphan Withdrawal Users,0
3,Orphan Tournament Users,0


In [8]:
# Check for missing demographics data
missing_demographics_query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(SystemAgeAsOfReg) AS missing_reg_dates,
    COUNT(*) - COUNT(CountryID) AS missing_countries,
    COUNT(*) - COUNT(Gender) AS missing_genders
FROM Demographics;
"""

missing_demographics_results = con.execute(missing_demographics_query).fetchdf()
missing_demographics_results

,total_rows,missing_user_ids,missing_reg_dates,missing_countries,missing_genders
0,5028,0,0,0,0


In [9]:

# Check for missing transaction data in CashGames
missing_transaction_data_query = """
SELECT 
    'CashGames' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(UserID) AS missing_user_ids,
    COUNT(*) - COUNT(Date) AS missing_dates,
    COUNT(*) - COUNT(Windows) AS missing_Windows,
    COUNT(*) - COUNT(StakesC) AS missing_StakesC,
    COUNT(*) - COUNT(WinningsC) AS missing_WinningsC
FROM CashGames
"""
missing_transaction_data = con.execute(missing_transaction_data_query).fetchdf()
missing_transaction_data

,table_name,total_rows,missing_user_ids,missing_dates,missing_Windows,missing_StakesC,missing_WinningsC
0,CashGames,51763,0,0,0,0,0


In [10]:
# Check for missing transaction data in Tournaments
missing_tournament_values = """
SELECT
    'tournament' as table_name,
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT("UserID") AS missingID,
    COUNT(*) - COUNT("Date") AS missingDate,
    COUNT(*) - COUNT("Trnmnts") AS missingTornaments,
    COUNT (*) - COUNT("StakesT") AS missingStakes,
    COUNT (*) - COUNT("WinningsT") as missingWinnings

FROM tournaments

"""

missing_transaction_data = con.execute(missing_tournament_values).fetch_df()
missing_transaction_data

,table_name,total_rows,missingID,missingDate,missingTornaments,missingStakes,missingWinnings
0,tournament,82831,0,0,0,0,0


In [11]:
# Check for missing transaction data in Deposits
missing_Deposit_query = """
SELECT
    'Deposits' as table_name,
    COUNT(*) - COUNT("UserID") AS missingID,
    COUNT(*) - COUNT("DepositID") AS missingDepositID,
    COUNT(*) - COUNT("SummaryDate") AS missingSummaryDate,
    COUNT(*) - COUNT("ProcessDate") AS missingProcessDate,
    COUNT(*) - COUNT("ProcessTime") AS missingProcessTime,
    COUNT(*) - COUNT("PayMeth") AS missingPayMeth,
    COUNT(*) - COUNT("PayMethCat") AS missingPayMethCat,
    COUNT(*) - COUNT("CardType") AS missingCardType,
    COUNT(*) - COUNT("Amount") AS missingAmount,
    COUNT(*) - COUNT("Status") AS missingStatus,
FROM deposits
"""

null_deposits_query = con.execute(missing_Deposit_query).fetch_df()
null_deposits_query

,table_name,missingID,missingDepositID,missingSummaryDate,missingProcessDate,missingProcessTime,missingPayMeth,missingPayMethCat,missingCardType,missingAmount,missingStatus
0,Deposits,0,0,0,0,0,0,0,9920,0,0


In [12]:
# Check for missing transaction data in Withdrawals
missing_withdrawals_query = """
SELECT
    'Deposits' as table_name,
    COUNT (*) AS total,
    COUNT(*) - COUNT("UserID") AS missingID,
    COUNT(*) - COUNT("WithdrawalID") AS missingWithdrawalID,
    COUNT(*) - COUNT("SummaryDate") AS missingSummaryDate,
    COUNT(*) - COUNT("ProcessDate") AS missingProcessDate,
    COUNT(*) - COUNT("ProcessTime") AS missingProcessTime,
    COUNT(*) - COUNT("PayMeth") AS missingPayMeth,
    COUNT(*) - COUNT("PayMethCat") AS missingPayMethCat,
    COUNT(*) - COUNT("CardType") AS missingCardType,
    COUNT(*) - COUNT("Amount") AS missingAmount,
    COUNT(*) - COUNT("Status") AS missingStatus,
FROM withdrawals
"""

missing_withdrawals_data = con.execute(missing_withdrawals_query).fetch_df()
missing_withdrawals_data

,table_name,total,missingID,missingWithdrawalID,missingSummaryDate,missingProcessDate,missingProcessTime,missingPayMeth,missingPayMethCat,missingCardType,missingAmount,missingStatus
0,Deposits,32307,0,0,0,0,0,0,0,1685,0,0


#### Phase 2 Summary & Data Health
- **Referential Integrity & Nulls:** 100% clean primary keys (`UserID`), zero orphan records, nulls documented for feature imputation.
- **Dataset Time Horizon:** 2015-02-01 to 2021-01-01.

## Data Preparation

### 3.1 Data Cleaning & Standardization
This step establishes a clean, standardized data layer across all raw CSV inputs. Primary keys are validated, text values are normalized, date strings are parsed into proper timestamps, and missing financial metrics are filled with zero values to prevent broken joins downstream.

In [13]:
# 3.1 Data Cleaning Layer Setup
cleaning_pipeline_query = """
-- 1. Clean Demographics
CREATE OR REPLACE VIEW Clean_Demographics AS
SELECT 
    CAST(UserID AS BIGINT) AS UserID,
    CAST(SystemAgeAsOfReg AS DATE) AS SystemAgeAsOfReg,
    COALESCE(UPPER(TRIM(CountryID)), 'UNKNOWN') AS CountryID,
    COALESCE(UPPER(TRIM(Gender)), 'U') AS Gender
FROM Demographics
WHERE UserID IS NOT NULL;

-- 2. Clean CashGames
CREATE OR REPLACE VIEW Clean_CashGames AS
SELECT 
    CAST(UserID AS BIGINT) AS UserID,
    CAST(Date AS TIMESTAMP) AS ActivityDate,
    COALESCE(CAST(StakesC AS DOUBLE), 0.0) AS StakesC,
    COALESCE(CAST(WinningsC AS DOUBLE), 0.0) AS WinningsC
FROM CashGames
WHERE UserID IS NOT NULL AND Date IS NOT NULL;

-- 3. Clean Tournaments
CREATE OR REPLACE VIEW Clean_Tournaments AS
SELECT 
    CAST(UserID AS BIGINT) AS UserID,
    CAST(Date AS TIMESTAMP) AS ActivityDate,
    COALESCE(CAST(Trnmnts AS DOUBLE), 0.0) AS Trnmnts,
    COALESCE(CAST(StakesT AS DOUBLE), 0.0) AS StakesT,
    COALESCE(CAST(WinningsT AS DOUBLE), 0.0) AS WinningsT
FROM Tournaments
WHERE UserID IS NOT NULL AND Date IS NOT NULL;

-- 4. Clean Deposits
CREATE OR REPLACE VIEW Clean_Deposits AS
SELECT 
    CAST(UserID AS BIGINT) AS UserID,
    CAST(ProcessDate AS TIMESTAMP) AS DepositDate,
    COALESCE(CAST(Amount AS DOUBLE), 0.0) AS DepositAmount
FROM Deposits
WHERE UserID IS NOT NULL AND ProcessDate IS NOT NULL;

-- 5. Clean Withdrawals
CREATE OR REPLACE VIEW Clean_Withdrawals AS
SELECT 
    CAST(UserID AS BIGINT) AS UserID,
    CAST(ProcessDate AS TIMESTAMP) AS WithdrawalDate,
    COALESCE(CAST(Amount AS DOUBLE), 0.0) AS WithdrawalAmount
FROM Withdrawals
WHERE UserID IS NOT NULL AND ProcessDate IS NOT NULL;
"""

con.execute(cleaning_pipeline_query)
print("Data cleaning layer initialized successfully across all 5 tables.")

Data cleaning layer initialized successfully across all 5 tables.


In [14]:
# 3.2 Temporal Bounds and Churn Label Construction (With Null Imputation)
temporal_target_query = """
-- Derive dataset timeline bounds dynamically
CREATE OR REPLACE VIEW Timeline_Bounds AS
WITH MaxDate AS (
    SELECT MAX(CAST(ActivityDate AS DATE)) AS T_max FROM Clean_CashGames
)
SELECT 
    T_max,
    T_max - INTERVAL '45 DAYS' AS T_cutoff
FROM MaxDate;

-- Construct player recency (X) and ground-truth churn labels (Y)
CREATE OR REPLACE VIEW Player_Temporal_Target AS
WITH Bounds AS (
    SELECT * FROM Timeline_Bounds
),
PreCutoffActivity AS (
    -- Most recent betting activity on or before T_cutoff
    SELECT 
        c.UserID,
        MAX(c.ActivityDate) AS LastBetDate
    FROM Clean_CashGames c, Bounds b
    WHERE c.ActivityDate <= b.T_cutoff
    GROUP BY c.UserID
),
PostCutoffActivity AS (
    -- Activity within the 45-day target evaluation window
    SELECT DISTINCT 
        c.UserID
    FROM Clean_CashGames c, Bounds b
    WHERE c.ActivityDate > b.T_cutoff AND c.ActivityDate <= b.T_max
)
SELECT 
    d.UserID,
    b.T_cutoff,
    pca.LastBetDate,
    -- Impute missing recency for non-bettors with a sentinel value (999 days)
    COALESCE(
        DATEDIFF('day', CAST(pca.LastBetDate AS DATE), b.T_cutoff), 
        999
    ) AS days_since_last_bet,
    CASE 
        WHEN post.UserID IS NULL THEN 1 
        ELSE 0 
    END AS churned
FROM Clean_Demographics d
CROSS JOIN Bounds b
LEFT JOIN PreCutoffActivity pca ON d.UserID = pca.UserID
LEFT JOIN PostCutoffActivity post ON d.UserID = post.UserID;
"""

results = con.execute(temporal_target_query)
con.execute("SELECT * FROM Player_Temporal_Target WHERE UserID = 344298;").fetchdf()

,UserID,T_cutoff,LastBetDate,days_since_last_bet,churned


In [15]:
con.execute("SELECT * FROM Player_Temporal_Target LIMIT 10;").fetchdf()

,UserID,T_cutoff,LastBetDate,days_since_last_bet,churned
0,34535,2020-06-13,2015-04-01,1900,0
1,38158,2020-06-13,2020-05-19,25,0
2,43093,2020-06-13,NaT,999,0
3,67724,2020-06-13,2020-06-04,9,0
4,3404,2020-06-13,2020-06-13,0,0
5,35311,2020-06-13,2020-03-30,75,0
6,39799,2020-06-13,2020-05-06,38,0
7,44298,2020-06-13,NaT,999,0
8,54178,2020-06-13,2020-05-25,19,0
9,56917,2020-06-13,2019-11-25,201,0


### 3.3 Feature Engineering & Aggregation

To build the predictive feature matrix ($X$), we aggregate player behavioral metrics across two primary domains—cash game activity and deposit transactions—using **strictly pre-cutoff telemetry** ($T \le T_{\text{cutoff}}$) to guarantee zero data leakage.

#### Feature Definitions & Engineering Rationale
* **Recency ($R$):** `days_since_last_bet` and `days_since_last_deposit` compute temporal distance from $T_{\text{cutoff}}$. Unobserved historical activity is explicitly imputed with a sentinel value of `999`.
* **Frequency ($F$):** `total_cash_sessions` (distinct active days) and `total_deposit_count` capture player engagement volume.
* **Monetary ($M$):** `total_stakes`, `total_winnings`, and `total_deposited` record cumulative financial transactions. `net_cash_flow` ($Winnings - Stakes$) models net player profitability.
* **Behavioral Ratios:**
  * `deposit_to_stake_ratio` ($\frac{\text{Deposits}}{\text{Stakes} + 1.0}$): Measures funding efficiency versus wagering activity.
  * `win_rate_ratio` ($\frac{\text{Winnings}}{\text{Stakes} + 1.0}$): Captures player return-on-investment (ROI). A smoothing factor of `+1.0` is added to denominators to prevent division-by-zero errors.
* **Null Handling:** All aggregated fields leverage `COALESCE` to enforce zero-null outputs across missing activity records.

In [16]:
# 3.3 Feature Matrix Construction (Pre-Cutoff Telemetry Only)
feature_matrix_query = """
CREATE OR REPLACE VIEW Player_Feature_Matrix AS
WITH Bounds AS (
    SELECT * FROM Timeline_Bounds
),

-- Aggregate Cash Game Metrics (Pre-Cutoff)
Cash_Stats AS (
    SELECT 
        c.UserID,
        COUNT(DISTINCT CAST(c.ActivityDate AS DATE)) AS total_cash_sessions,
        SUM(c.StakesC) AS total_stakes,
        SUM(c.WinningsC) AS total_winnings
    FROM Clean_CashGames c, Bounds b
    WHERE c.ActivityDate <= b.T_cutoff
    GROUP BY c.UserID
),

-- Aggregate Deposit Metrics (Pre-Cutoff)
Deposit_Stats AS (
    SELECT 
        d.UserID,
        COUNT(*) AS total_deposit_count,
        SUM(d.DepositAmount) AS total_deposited,
        MAX(d.DepositDate) AS LastDepositDate
    FROM Clean_Deposits d, Bounds b
    WHERE d.DepositDate <= b.T_cutoff
    GROUP BY d.UserID
)

-- Main Feature Matrix Join & Imputation
SELECT 
    t.UserID,
    t.days_since_last_bet,
    t.churned,
    
    -- Cash Game Features
    COALESCE(cs.total_cash_sessions, 0) AS total_cash_sessions,
    COALESCE(cs.total_stakes, 0.0) AS total_stakes,
    COALESCE(cs.total_winnings, 0.0) AS total_winnings,
    COALESCE(cs.total_winnings - cs.total_stakes, 0.0) AS net_cash_flow,
    
    -- Deposit Features
    COALESCE(ds.total_deposit_count, 0) AS total_deposit_count,
    COALESCE(ds.total_deposited, 0.0) AS total_deposited,
    COALESCE(
        DATEDIFF('day', CAST(ds.LastDepositDate AS DATE), b.T_cutoff), 
        999
    ) AS days_since_last_deposit,
    
    -- Risk & Velocity Ratios (+1.0 denominator prevents DivisionByZero)
    COALESCE(ds.total_deposited / (cs.total_stakes + 1.0), 0.0) AS deposit_to_stake_ratio,
    COALESCE(cs.total_winnings / (cs.total_stakes + 1.0), 0.0) AS win_rate_ratio

FROM Player_Temporal_Target t
CROSS JOIN Bounds b
LEFT JOIN Cash_Stats cs ON t.UserID = cs.UserID
LEFT JOIN Deposit_Stats ds ON t.UserID = ds.UserID;
"""

con.execute(feature_matrix_query)
print("Player_Feature_Matrix view created successfully.")
con.execute("SELECT * FROM Player_Feature_Matrix LIMIT 5;").fetchdf()

Player_Feature_Matrix view created successfully.


,UserID,days_since_last_bet,churned,total_cash_sessions,total_stakes,total_winnings,net_cash_flow,total_deposit_count,total_deposited,days_since_last_deposit,deposit_to_stake_ratio,win_rate_ratio
0,69225,1498,1,4,95.31,60.31,-35.00,883,33218.00,0,344.907071,0.626207
1,9896,1943,1,1,3.51,0.00,-3.51,1053,13070.00,0,2898.004435,0.000000
2,19417,182,1,49,1551.66,1289.48,-262.18,356,6593.40,94,4.246519,0.830497
3,18124,1663,1,3,212.56,146.22,-66.34,106,2678.64,0,12.542798,0.684679
4,16481,631,0,61,1137.35,956.67,-180.68,605,6565.00,1,5.767119,0.840401


In [17]:
con.execute("SELECT * FROM Player_Feature_Matrix LIMIT 5;").fetchdf()

,UserID,days_since_last_bet,churned,total_cash_sessions,total_stakes,total_winnings,net_cash_flow,total_deposit_count,total_deposited,days_since_last_deposit,deposit_to_stake_ratio,win_rate_ratio
0,46725,35,0,154,9511.91,8549.73,-962.18,263,6329.40,33,0.665348,0.898750
1,51366,449,1,5,2982.03,2265.50,-716.53,384,14988.06,14,5.024442,0.759463
2,4780,999,0,0,0.00,0.00,0.00,74,1480.00,1,0.000000,0.000000
3,71632,1707,1,2,7.85,6.58,-1.27,337,3693.80,107,417.378531,0.743503
4,47855,250,1,1,25.17,25.10,-0.07,53,530.00,105,20.252197,0.959113


### 3.4 Final Dataset Export & Quality Audit

Before transitioning to Phase 4 (Modeling), we extract `Player_Feature_Matrix` into Pandas, audit missing values, verify target distribution class imbalance, and export the sanitized feature set to disk.

In [18]:
# 3.4 Extract DataFrame & Perform Integrity Audit
import pandas as pd

# 1. Fetch full feature dataset
df_model = con.execute("SELECT * FROM Player_Feature_Matrix;").fetchdf()

# 2. Dataset Shape & Null Audit
print("--- DATASET INTEGRITY CHECK ---")
print(f"Total Rows (Players): {len(df_model):,}")
print(f"Total Columns: {len(df_model.columns)}")
print(f"Total Null Values Across Matrix: {df_model.isnull().sum().sum()}")

# 3. Target Distribution Check (Class Imbalance Analysis)
churn_counts = df_model['churned'].value_counts()
churn_pct = df_model['churned'].value_counts(normalize=True) * 100

print("\n--- CHURN TARGET DISTRIBUTION (Y) ---")
print(f"Retained (0): {churn_counts.get(0, 0):,} ({churn_pct.get(0, 0):.2f}%)")
print(f"Churned  (1): {churn_counts.get(1, 0):,} ({churn_pct.get(1, 0):.2f}%)")

# 4. Export Clean Model Dataset
df_model.to_parquet("igaming_churn_prepared_dataset.parquet", index=False)
print("\nSuccess: Final modeling dataset saved to 'igaming_churn_prepared_dataset.parquet'.")

# Preview final schema
df_model.head()

--- DATASET INTEGRITY CHECK ---
Total Rows (Players): 5,028
Total Columns: 12
Total Null Values Across Matrix: 0

--- CHURN TARGET DISTRIBUTION (Y) ---
Retained (0): 40 (0.80%)
Churned  (1): 4,988 (99.20%)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [19]:
# Diagnostic Query: Check Activity Distribution over time
con.execute("""
SELECT 
    MIN(CAST(ActivityDate AS DATE)) AS EarliestDate,
    MAX(CAST(ActivityDate AS DATE)) AS LatestDate,
    COUNT(DISTINCT UserID) AS TotalUsers,
    COUNT(*) AS TotalTransactions
FROM Clean_CashGames;
""").fetchdf()

,EarliestDate,LatestDate,TotalUsers,TotalTransactions
0,2015-02-01,2020-07-28,4232,51763


In [20]:
con.execute("""
SELECT 
    STRFTIME(CAST(ActivityDate AS DATE), '%Y-%m') AS YearMonth,
    COUNT(DISTINCT UserID) AS ActiveUsers,
    COUNT(*) AS TotalBets
FROM Clean_CashGames
GROUP BY YearMonth
ORDER BY YearMonth ASC;
""").fetchdf()

,YearMonth,ActiveUsers,TotalBets
0,2015-02,2686,8546
1,2015-03,1536,7439
2,2015-04,729,3767
3,2015-05,524,2765
4,2015-06,384,1963
...,...,...,...
61,2020-03,57,168
62,2020-04,45,150
63,2020-05,47,130
64,2020-06,30,81
